# Literature And IPD Query Notebook Showcase

This notebook is organized for presentation, with two main goals:

1. **Query literature evidence** extracted from papers, PDFs, and URLs.
2. **Query IPD data** from the vertical ROI dataset.

Use this notebook to show your supervisor:

- how literature sources are converted into a structured database
- how disease-related features can be searched from literature
- how subject-level IPD rows can be filtered by diagnosis, ROI, metric, and sex


## Data Sources

This workflow depends on two external data sources that are **not committed** to this repository:

1. **Literature sources**: local PDF files and/or public URLs configured in `INPUTS`.
   Typical examples include PubMed pages, PMC articles, and local papers placed in the Neurodesk workspace.
2. **IPD ROI table**: the vertical CSV file referenced by `IPD_CSV_PATH`, currently expected at:
   `/home/jovyan/oasis_selected/ipd_vertical_dataset/roi_vertical_dataset_template.csv`

Generated database artifacts such as `literature_database.sql`, `literature_database.json`, and `literature_database.xlsx`
are workflow outputs and should be regenerated locally instead of uploaded.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

project_dir = Path.cwd()
sys.path.append(str(project_dir))

from neurodesk_literature_to_pgsql import analyze_sources


## 1. Configure Data Sources

### Set literature inputs and IPD table path

This section defines:

- which PDFs or URLs should be used as literature sources
- where the structured literature outputs should be saved
- where the IPD vertical dataset CSV is located

For GitHub upload, keep the source configuration here but do not commit the generated database files or private datasets.


In [ ]:
# LITERATURE SOURCES
INPUTS = [
    # '/home/jovyan/Desktop/paper.pdf',
    # 'https://pubmed.ncbi.nlm.nih.gov/32711083/',
    # 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10406233/',
]

# LITERATURE OUTPUT DIRECTORY
OUTPUT_DIR = Path('./literature_outputs_workflow_demo')

# OPTIONAL: if INPUTS is empty, the notebook will try to load this existing JSON file
LITERATURE_JSON_PATH = OUTPUT_DIR / 'literature_database.json'

# IPD DATASET LOCATION
IPD_CSV_PATH = Path('/home/jovyan/oasis_selected/ipd_vertical_dataset/roi_vertical_dataset_template.csv')

# AI BACKEND SETTINGS
AI_BACKEND = 'none'
AI_BASE_URL = 'http://127.0.0.1:11434/v1'
AI_MODEL = 'llama3.1:8b'
AI_API_KEY = None

display(pd.DataFrame({
    'setting': ['n_inputs', 'output_dir', 'literature_json_path', 'ipd_csv_path', 'ai_backend'],
    'value': [len(INPUTS), str(OUTPUT_DIR), str(LITERATURE_JSON_PATH), str(IPD_CSV_PATH), AI_BACKEND],
} ))


## 2. Build Or Load The Literature Database

### Create fresh outputs or load an existing literature JSON file

Logic used here:

- If `INPUTS` contains sources, the extraction pipeline is run.
- If `INPUTS` is empty, the notebook loads an existing `literature_database.json`.
- This makes it easier to demonstrate queries without rerunning extraction every time.


In [ ]:
result = None

if INPUTS:
    result = analyze_sources(
        inputs=INPUTS,
        output_dir=OUTPUT_DIR,
        ai_backend=AI_BACKEND,
        ai_base_url=AI_BASE_URL,
        ai_model=AI_MODEL,
        ai_api_key=AI_API_KEY,
    )
    records = result['records']
else:
    if not LITERATURE_JSON_PATH.exists():
        raise FileNotFoundError(
            f'No INPUTS were provided and no existing literature JSON was found at {LITERATURE_JSON_PATH}'
        )
    records = json.loads(LITERATURE_JSON_PATH.read_text(encoding='utf-8'))

records_df = pd.DataFrame(records)

display(pd.DataFrame([
    {'item': 'literature_record_count', 'value': len(records)},
    {'item': 'used_existing_json', 'value': result is None},
    {'item': 'literature_json_path', 'value': str(LITERATURE_JSON_PATH if result is None else result['json_path'])},
    {'item': 'ipd_csv_exists', 'value': IPD_CSV_PATH.exists()},
]))


## 3. Preview The Literature Database

### Inspect the main literature fields before querying

This preview is useful in meetings because it shows that the database is already structured into fields such as:

- `diagnoses`
- `symptoms`
- `rois`
- `imaging_metrics`
- `source_type`


In [ ]:
literature_preview_cols = [
    'title',
    'year',
    'doi',
    'diagnoses',
    'symptoms',
    'rois',
    'imaging_metrics',
    'sex_scope',
    'source_type',
]

display(records_df[[c for c in literature_preview_cols if c in records_df.columns]])


## 4. How To Query Literature Data

### Search disease-related evidence from the literature database

These helper functions support three common query styles:

1. find all literature rows matching a disease / ROI / metric
2. list features reported for one disease
3. reverse-search likely diseases from one or more features


In [ ]:
def _list_contains(values, target):
    target_lower = target.lower()
    return any(str(v).lower() == target_lower for v in (values or []))


def _list_overlap(values, wanted_values):
    wanted = {str(v).lower() for v in wanted_values}
    return [v for v in (values or []) if str(v).lower() in wanted]


def query_literature(records, disease=None, roi=None, metric=None, modality=None, source_type=None, title_contains=None):
    rows = []
    for rec in records:
        if disease and not _list_contains(rec.get('diagnoses'), disease):
            continue
        if roi and not _list_contains(rec.get('rois'), roi):
            continue
        if metric and not _list_contains(rec.get('imaging_metrics'), metric):
            continue
        if modality and not _list_contains(rec.get('modalities'), modality):
            continue
        if source_type and str(rec.get('source_type', '')).lower() != source_type.lower():
            continue
        if title_contains and title_contains.lower() not in str(rec.get('title', '')).lower():
            continue
        rows.append(rec)
    return pd.DataFrame(rows)


def literature_features_for_disease(records, disease):
    rows = []
    seen = set()
    for rec in records:
        if not _list_contains(rec.get('diagnoses'), disease):
            continue
        for feature in rec.get('symptoms') or []:
            key = (str(feature).lower(), rec.get('title'))
            if key in seen:
                continue
            seen.add(key)
            rows.append({
                'disease': disease,
                'feature': feature,
                'paper_title': rec.get('title'),
                'doi': rec.get('doi'),
                'source_type': rec.get('source_type'),
            })
    return pd.DataFrame(rows)


def literature_diseases_for_features(records, feature_list):
    rows = []
    matched = {}
    wanted = {str(v).lower() for v in feature_list}
    for rec in records:
        overlap = _list_overlap(rec.get('symptoms'), wanted)
        if not overlap:
            continue
        for disease in rec.get('diagnoses') or []:
            item = matched.setdefault(disease, {'features': set(), 'papers': set()})
            item['features'].update(overlap)
            if rec.get('title'):
                item['papers'].add(rec['title'])
    for disease, payload in matched.items():
        rows.append({
            'disease': disease,
            'matched_features': sorted(payload['features']),
            'match_count': len(payload['features']),
            'source_papers': sorted(payload['papers']),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(['match_count', 'disease'], ascending=[False, True]).reset_index(drop=True)
    return out


In [ ]:
display(Markdown('### Literature query example A: AD-related rows'))
display(query_literature(records, disease='AD'))

display(Markdown('### Literature query example B: Features reported for Dementia'))
display(literature_features_for_disease(records, 'Dementia'))

display(Markdown('### Literature query example C: Diseases linked to cortical thickness and hippocampus'))
display(literature_diseases_for_features(records, ['cortical thickness', 'hippocampus']))


## 5. How To Query IPD Data

### Search subject-level rows from the vertical ROI dataset

This section is for the subject-level table, not the literature table.

Typical IPD questions are:

- which subjects belong to one diagnosis group?
- which rows correspond to `hippocampus` or `cortical_thickness`?
- are there sex-specific rows or group differences?
- which values are still missing and need FreeSurfer output?


In [ ]:
if not IPD_CSV_PATH.exists():
    raise FileNotFoundError(f'IPD CSV not found: {IPD_CSV_PATH}')

ipd_df = pd.read_csv(IPD_CSV_PATH)

display(pd.DataFrame([
    {'item': 'ipd_row_count', 'value': len(ipd_df)},
    {'item': 'ipd_column_count', 'value': len(ipd_df.columns)},
    {'item': 'ipd_csv_path', 'value': str(IPD_CSV_PATH)},
]))

display(ipd_df.head())


In [ ]:
def query_ipd(df, diagnosis=None, roi=None, metric=None, sex=None, data_level=None, statistic_type=None):
    out = df.copy()
    if diagnosis is not None and 'diagnosis' in out.columns:
        out = out[out['diagnosis'].astype(str).str.lower() == diagnosis.lower()]
    if roi is not None and 'roi_name' in out.columns:
        out = out[out['roi_name'].astype(str).str.lower() == roi.lower()]
    if metric is not None and 'imaging_metric' in out.columns:
        out = out[out['imaging_metric'].astype(str).str.lower() == metric.lower()]
    if sex is not None and 'sex' in out.columns:
        out = out[out['sex'].astype(str).str.lower() == sex.lower()]
    if data_level is not None and 'data_level' in out.columns:
        out = out[out['data_level'].astype(str).str.lower() == data_level.lower()]
    if statistic_type is not None and 'statistic_type' in out.columns:
        out = out[out['statistic_type'].astype(str).str.lower() == statistic_type.lower()]
    return out.reset_index(drop=True)


def summarize_ipd(df, group_cols=('diagnosis', 'roi_name', 'imaging_metric')):
    needed = [c for c in group_cols if c in df.columns]
    work = df.copy()
    if 'value_numeric' in work.columns:
        work['value_numeric'] = pd.to_numeric(work['value_numeric'], errors='coerce')
    summary = work.groupby(needed, dropna=False).agg(
        n_rows=('value_numeric', 'size'),
        n_non_missing=('value_numeric', lambda s: s.notna().sum()),
        mean_value=('value_numeric', 'mean'),
    ).reset_index()
    return summary


def missing_ipd_values(df):
    out = df.copy()
    if 'value_numeric' in out.columns:
        out['value_numeric'] = pd.to_numeric(out['value_numeric'], errors='coerce')
        out = out[out['value_numeric'].isna()]
    return out.reset_index(drop=True)


In [ ]:
display(Markdown('### IPD query example A: Dementia rows'))
display(query_ipd(ipd_df, diagnosis='Dementia').head(20))

display(Markdown('### IPD query example B: Hippocampus ROI volume rows'))
display(query_ipd(ipd_df, roi='hippocampus', metric='roi_volume').head(20))

display(Markdown('### IPD query example C: Summary by diagnosis, ROI, and metric'))
display(summarize_ipd(ipd_df).head(20))

display(Markdown('### IPD query example D: Rows still missing numeric values'))
display(missing_ipd_values(ipd_df).head(20))


## 6. Suggested Presentation Order

### Recommended sequence for supervisor discussion

A clean presentation sequence is:

1. Show the configured literature sources and IPD path.
2. Show that the literature database has already been built or loaded.
3. Demonstrate one literature query for a disease such as `AD` or `Dementia`.
4. Demonstrate one reverse literature query from features to diseases.
5. Demonstrate one IPD filter, for example `diagnosis='Dementia'` and `roi='hippocampus'`.
6. End with the missing-value table to explain what still depends on FreeSurfer outputs.

This makes the workflow easy to explain:

- literature database -> disease evidence lookup
- IPD table -> subject-level ROI lookup
